# NLP Evaluation Module — Interview Practice Platform
### Evaluates candidate answers using:
- **BERTScore F1** — Semantic similarity to golden answer
- **GPT-2 Perplexity** — Grammar & fluency scoring
- **Keyword Coverage Score** — Technical term coverage

**Supported Rounds:** Technical | HR

**LLM Backend:** Groq API (Llama 3.3 70B) — free, fast, no rate limit issues

**Flow:** Groq generates Question + Golden Answer → Candidate inputs answer → NLP scores computed

---
### Metric Reference (from project spec)

| BERTScore F1 | Interpretation |
|---|---|
| 0.90 – 1.00 | Excellent semantic match |
| 0.75 – 0.89 | Strong conceptual similarity |
| 0.60 – 0.74 | Partial conceptual coverage |
| < 0.60 | Weak or irrelevant answer |

| Perplexity | Linguistic Quality |
|---|---|
| < 20 | Highly fluent |
| 20 – 40 | Good grammar |
| 40 – 60 | Acceptable quality |
| > 60 | Poor grammar or unnatural phrasing |

## Cell 2: Install Dependencies

In [ ]:
!pip install groq bert-score transformers torch nltk -q

## Cell 3: Imports & Setup

In [ ]:
import torch
import math
import re
import warnings
import nltk
from groq import Groq
from bert_score import score as bert_score
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

warnings.filterwarnings('ignore')
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

print('All imports successful')
print(f'Device: {"GPU" if torch.cuda.is_available() else "CPU"}')

## Cell 4: Configure Groq API


In [ ]:
GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERE"

groq_client = Groq(api_key=GROQ_API_KEY)
GROQ_MODEL  = 'llama-3.3-70b-versatile' 

def call_groq(prompt: str) -> str:
    """
    Calls Groq API (Llama 3.3 70B) to generate Q&A pairs.
    Groq free tier: 30 req/min — fast enough to run back-to-back without sleeps.
    """
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                'role': 'system',
                'content': (
                    'You are an expert interview coach. '
                    'Always respond in the exact format requested. '
                    'Do not add any extra text, preamble, or explanation.'
                )
            },
            {'role': 'user', 'content': prompt}
        ],
        model=GROQ_MODEL,
        temperature=0.7,
        max_tokens=512,
    )
    return chat_completion.choices[0].message.content.strip()

test = call_groq('Say OK')
print(f'Groq connected successfully. Test response: "{test}"')

## Cell 5: Load GPT-2 for Perplexity Scoring

In [ ]:
print('Loading GPT-2 model...')

gpt2_tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
gpt2_model     = GPT2LMHeadModel.from_pretrained('gpt2')
gpt2_model.eval()

print('GPT-2 loaded successfully')

## Cell 6: Scoring Functions

In [ ]:
# ── 1. BERTScore — Semantic Similarity ──────────────────────────────────
def compute_bertscore(candidate: str, reference: str) -> float:
    """
    Computes BERTScore F1 between candidate and golden answer.
    Formula: F1 = 2 * Precision * Recall / (Precision + Recall)
    Uses contextual embeddings — captures paraphrasing unlike BLEU/ROUGE.
    Returns float in [0, 1].
    """
    P, R, F1 = bert_score(
        [candidate], [reference],
        lang='en',
        model_type='distilbert-base-uncased',
        verbose=False
    )
    return round(F1[0].item(), 4)


def interpret_bertscore(score: float) -> str:
    """Interpretation labels per project spec Table 1."""
    if score >= 0.90: return 'Excellent semantic match'
    elif score >= 0.75: return 'Strong conceptual similarity'
    elif score >= 0.60: return 'Partial conceptual coverage'
    else: return 'Weak or irrelevant answer'


# ── 2. GPT-2 Perplexity — Fluency & Grammar ──────────────────────────────
def compute_perplexity(text: str) -> float:
    """
    Computes GPT-2 perplexity. Lower = more fluent and grammatically natural.
    Perplexity = 2^H(p) where H(p) is cross-entropy of predicted distribution.
    """
    encodings = gpt2_tokenizer(text, return_tensors='pt')
    input_ids = encodings.input_ids
    max_len = gpt2_model.config.n_positions
    if input_ids.shape[1] > max_len:
        input_ids = input_ids[:, :max_len]
    with torch.no_grad():
        outputs = gpt2_model(input_ids, labels=input_ids)
    return round(math.exp(outputs.loss.item()), 2)


def interpret_perplexity(perplexity: float) -> str:
    """Interpretation labels per project spec Table 2."""
    if perplexity < 20: return 'Highly fluent'
    elif perplexity < 40: return 'Good grammar'
    elif perplexity < 60: return 'Acceptable quality'
    else: return 'Poor grammar or unnatural phrasing'


def perplexity_to_fluency_score(perplexity: float) -> float:
    """
    Normalises raw perplexity to [0, 1] fluency score.
    Thresholds from project spec Table 2:
      < 20  -> Highly fluent    -> 0.90 - 1.00
      20-40 -> Good grammar     -> 0.70 - 0.90
      40-60 -> Acceptable       -> 0.45 - 0.70
      > 60  -> Poor             -> 0.00 - 0.45
    """
    if perplexity < 20:
        return round(1.0 - (perplexity / 20) * 0.10, 4)
    elif perplexity < 40:
        return round(0.90 - ((perplexity - 20) / 20) * 0.20, 4)
    elif perplexity < 60:
        return round(0.70 - ((perplexity - 40) / 20) * 0.25, 4)
    else:
        return round(max(0.0, 0.45 - ((perplexity - 60) / 200) * 0.45), 4)


# ── 3. Keyword Coverage — Technical Depth ────────────────────────────────
def extract_technical_keywords(text: str) -> set:
    """Extracts meaningful keywords (stopwords removed)."""
    from nltk.corpus import stopwords
    stop_words = set(stopwords.words('english'))
    tokens = re.findall(r'\b[a-zA-Z][a-zA-Z0-9_\-]{2,}\b', text.lower())
    return {t for t in tokens if t not in stop_words}


def compute_keyword_coverage(candidate: str, reference: str) -> float:
    """
    TechnicalScore = Correct Keywords Used / Total Expected Keywords
    Returns float in [0, 1].
    """
    ref_kws  = extract_technical_keywords(reference)
    cand_kws = extract_technical_keywords(candidate)
    if not ref_kws:
        return 1.0
    covered = ref_kws.intersection(cand_kws)
    return round(len(covered) / len(ref_kws), 4)


print('Scoring functions defined')

## Cell 7: Groq Q&A Generator

In [ ]:
def generate_qa_pair(round_type: str, topic: str = None) -> dict:
    """
    Uses Groq (Llama 3.3 70B) to generate a question + golden answer.
    round_type : 'technical' or 'hr'
    topic      : optional hint e.g. 'data structures', 'leadership'
    """
    if round_type not in ('technical', 'hr'):
        raise ValueError(f"round_type must be 'technical' or 'hr', got '{round_type}'")

    if round_type == 'technical':
        topic_hint = f'about {topic}' if topic else 'covering a core CS/engineering concept'
        prompt = (
            f'Generate one technical interview question {topic_hint}.\n'
            'Respond in EXACTLY this format, no extra text:\n'
            'QUESTION: <the interview question>\n'
            'ANSWER: <a comprehensive golden answer an expert candidate would give, 3-6 sentences>\n'
            'KEYWORDS: <comma-separated list of 5-10 must-have technical keywords from the answer>'
        )
    else:
        topic_hint = f'about {topic}' if topic else 'on a common HR/behavioural topic'
        prompt = (
            f'Generate one HR interview question {topic_hint}.\n'
            'Respond in EXACTLY this format, no extra text:\n'
            'QUESTION: <the interview question>\n'
            'ANSWER: <a strong golden answer using the STAR method, 3-6 sentences>\n'
            'KEYWORDS: <comma-separated list of 5-8 key behavioural/soft-skill keywords>'
        )

    text = call_groq(prompt)

    result = {'round_type': round_type}
    for line in text.split('\n'):
        line = line.strip()
        if line.startswith('QUESTION:'):
            result['question'] = line.replace('QUESTION:', '').strip()
        elif line.startswith('ANSWER:'):
            result['golden_answer'] = line.replace('ANSWER:', '').strip()
        elif line.startswith('KEYWORDS:'):
            kws = line.replace('KEYWORDS:', '').strip()
            result['keywords'] = [k.strip() for k in kws.split(',')]

    return result


print('Groq Q&A generator ready')

## Cell 8: Main Evaluation Pipeline

In [ ]:
def evaluate_candidate_answer(
    candidate_answer: str,
    golden_answer: str,
    round_type: str,
    question: str = '',
    weights: dict = None
) -> dict:
    """
    Full NLP evaluation pipeline.
    FinalScore = w1(BERTScore) + w2(FluencyScore) + w3(TechnicalScore)

    Default weights:
      Technical : semantic=0.45, fluency=0.20, keyword=0.35
      HR        : semantic=0.55, fluency=0.35, keyword=0.10
    """
    if weights is None:
        weights = {'semantic': 0.45, 'fluency': 0.20, 'keyword': 0.35} if round_type == 'technical' \
             else {'semantic': 0.55, 'fluency': 0.35, 'keyword': 0.10}

    print('\n' + '=' * 60)
    print(f'  Round Type : {round_type.upper()}')
    if question:
        q_display = question[:80] + '...' if len(question) > 80 else question
        print(f'  Question   : {q_display}')
    print('=' * 60)

    # Score 1: BERTScore
    print('\nComputing BERTScore (semantic similarity)...')
    bert_f1 = compute_bertscore(candidate_answer, golden_answer)
    print(f'  BERTScore F1    : {bert_f1:.4f}  -> {interpret_bertscore(bert_f1)}')

    # Score 2: Perplexity
    print('Computing GPT-2 perplexity (fluency/grammar)...')
    perplexity    = compute_perplexity(candidate_answer)
    fluency_score = perplexity_to_fluency_score(perplexity)
    print(f'  Raw Perplexity  : {perplexity:.2f}   -> {interpret_perplexity(perplexity)}')
    print(f'  Fluency Score   : {fluency_score:.4f}')

    # Score 3: Keyword Coverage
    print('Computing keyword coverage...')
    keyword_score = compute_keyword_coverage(candidate_answer, golden_answer)
    ref_kws  = extract_technical_keywords(golden_answer)
    cand_kws = extract_technical_keywords(candidate_answer)
    covered  = ref_kws.intersection(cand_kws)
    missing  = ref_kws - cand_kws
    print(f'  Keyword Coverage: {keyword_score:.4f}  ({len(covered)}/{len(ref_kws)} keywords covered)')
    if missing:
        print(f'  Missing keywords: {", ".join(list(missing)[:8])}')

    # Weighted Final Score
    final_score = (
        weights['semantic'] * bert_f1 +
        weights['fluency']  * fluency_score +
        weights['keyword']  * keyword_score
    )

    # Grade aligned with BERTScore interpretation bands from spec
    if   final_score >= 0.90: grade = 'Excellent'
    elif final_score >= 0.75: grade = 'Strong'
    elif final_score >= 0.60: grade = 'Partial'
    else:                     grade = 'Weak'

    print('\n' + '-' * 60)
    print(f'  SCORE BREAKDOWN  (weights: {weights})')
    print(f'    Semantic  : {bert_f1:.4f} x {weights["semantic"]} = {weights["semantic"]*bert_f1:.4f}')
    print(f'    Fluency   : {fluency_score:.4f} x {weights["fluency"]} = {weights["fluency"]*fluency_score:.4f}')
    print(f'    Keywords  : {keyword_score:.4f} x {weights["keyword"]} = {weights["keyword"]*keyword_score:.4f}')
    print(f'    {"-" * 38}')
    print(f'    FINAL NLP SCORE : {final_score:.4f}  -> {grade}')
    print('-' * 60)

    return {
        'question': question, 'round_type': round_type,
        'bert_f1': bert_f1, 'perplexity': perplexity,
        'fluency_score': fluency_score, 'keyword_score': keyword_score,
        'covered_keywords': list(covered), 'missing_keywords': list(missing),
        'weights': weights, 'final_score': round(final_score, 4), 'grade': grade
    }


print('Evaluation pipeline ready')

## Cell 9: Run — Technical Round Demo

In [ ]:
# Step 1: Generate Q&A from Groq
print('Generating technical question via Groq...\n')
tech_qa = generate_qa_pair(round_type='technical', topic='data structures')

print(f"Question      : {tech_qa.get('question', 'N/A')}")
print(f"Golden Answer : {tech_qa.get('golden_answer', 'N/A')}")
print(f"Keywords      : {tech_qa.get('keywords', [])}")

In [ ]:

candidate_answer_tech = input('Enter candidate answer (or press Enter for demo): ').strip()

if not candidate_answer_tech:
    candidate_answer_tech = (
        'A hash table stores data using key value pairs. '
        'It uses a hash function to compute an index into an array. '
        'Collisions can happen when two keys map to same index, '
        'which we handle using chaining or open addressing.'
    )
    print(f'[Demo] Using sample answer: "{candidate_answer_tech}"')

# Step 3: Evaluate
tech_results = evaluate_candidate_answer(
    candidate_answer=candidate_answer_tech,
    golden_answer=tech_qa.get('golden_answer', ''),
    round_type='technical',
    question=tech_qa.get('question', '')
)

## Cell 10: Run — HR Round Demo

In [ ]:
# Step 1: Generate HR Q&A from Groq
print('Generating HR question via Groq...\n')
hr_qa = generate_qa_pair(round_type='hr', topic='conflict resolution')

print(f"Question      : {hr_qa.get('question', 'N/A')}")
print(f"Golden Answer : {hr_qa.get('golden_answer', 'N/A')}")
print(f"Keywords      : {hr_qa.get('keywords', [])}")

In [ ]:
# Step 2: Enter candidate answer
candidate_answer_hr = input('Enter candidate answer (or press Enter for demo): ').strip()

if not candidate_answer_hr:
    candidate_answer_hr = (
        'When I faced a conflict with a teammate over project priorities, '
        'I arranged a one on one meeting to listen to their concerns. '
        'We found common ground by focusing on the shared team goal. '
        'The outcome was a revised timeline that worked for both of us and the project delivered on time.'
    )
    print(f'[Demo] Using sample answer: "{candidate_answer_hr}"')

# Step 3: Evaluate
hr_results = evaluate_candidate_answer(
    candidate_answer=candidate_answer_hr,
    golden_answer=hr_qa.get('golden_answer', ''),
    round_type='hr',
    question=hr_qa.get('question', '')
)

## Cell 11: Summary Report

In [ ]:
def print_summary_report(results_list: list):
    print('\n' + '=' * 72)
    print('  CANDIDATE EVALUATION SUMMARY REPORT')
    print('=' * 72)
    print(f"  {'#':<3} {'Round':<12} {'BERTScore':<12} {'Fluency':<10} {'Keywords':<12} {'Final':<8} Grade")
    print('  ' + '-' * 68)
    total = 0
    for i, r in enumerate(results_list, 1):
        print(
            f"  {i:<3} {r['round_type'].upper():<12}"
            f"{r['bert_f1']:<12.4f}"
            f"{r['fluency_score']:<10.4f}"
            f"{r['keyword_score']:<12.4f}"
            f"{r['final_score']:<8.4f}"
            f"{r['grade']}"
        )
        total += r['final_score']
    avg = total / len(results_list)
    print('  ' + '-' * 68)
    print(f"  {'OVERALL AVERAGE':<46} {avg:.4f}")
    print('=' * 72)
    print()
    print('  NOTE: These NLP scores feed into the platform alongside')
    print('        Audio module scores  (prosody, pace, confidence)')
    print('        CV module scores     (facial expressions, eye contact)')
    print('=' * 72)


all_results = [tech_results, hr_results]
print_summary_report(all_results)

## Cell 12: Integration Hook (for Audio Module)

In [ ]:
def run_nlp_pipeline(transcribed_text: str, round_type: str, topic: str = None) -> dict:
    """
    Entry point for the audio module integration.

    Parameters
    ----------
    transcribed_text : str  -- output of Whisper/ASR from audio module
    round_type       : str  -- 'technical' or 'hr'
    topic            : str  -- optional topic for Q generation

    Returns
    -------
    dict -- full evaluation results
    """
    qa = generate_qa_pair(round_type=round_type, topic=topic)
    results = evaluate_candidate_answer(
        candidate_answer=transcribed_text,
        golden_answer=qa.get('golden_answer', ''),
        round_type=round_type,
        question=qa.get('question', '')
    )
    results['golden_answer']     = qa.get('golden_answer', '')
    results['expected_keywords'] = qa.get('keywords', [])
    return results


print('Integration hook run_nlp_pipeline() is ready.')
print()
print('Usage from audio module:')
print('  results = run_nlp_pipeline(')
print('      transcribed_text = asr_output,')
print('      round_type       = "technical",')
print('      topic            = "operating systems"')
print('  )')